In [1]:
"""
Generate the solution for any YAML file in problem's directory, storing the processing time and cost in a PERF_FILE.
In addition, it can compare the results with a reference and show the differences.
"""
import csv
from os import path, getcwd, chdir
from glob import glob
from time import time
from fcma import Vm
from ascal import Ascal
from ascal import AscalConfig
import aws_eu_west_1_c5m5r5

PROBLEMS_DIR = "../tests/problems"
PERF_FILE = "ascal_perf.csv"
REF_PERF_FILE = "ascal_perf_ref.csv"
PER_FILE_FIELDS = ["problem_file", "processing_time(s)", "cost($)", "speedup", "cost_divider"]
TOTAL = "TOTAL"

# List of problem YAML files
problems_dir = f"{getcwd()}/{PROBLEMS_DIR}"
problem_files = yaml_files = glob(path.join(problems_dir, "*.yaml"))

# Check if reference performance file exists and prepare for comparison
ref_perf_data = {}
if path.exists(REF_PERF_FILE):
    total_ref_processing_time = 0.0
    total_ref_cost = 0.0
    with open(REF_PERF_FILE, 'r', newline="") as ref_perf_file:
        reader = csv.DictReader(ref_perf_file)
        for row in reader:
            ref_perf_data[row["problem_file"]] = {
                "processing_time": float(row["processing_time(s)"]),
                "cost": float(row["cost($)"])
            }
    total_ref_processing_time = ref_perf_data[TOTAL]["processing_time"]
    total_ref_cost = ref_perf_data[TOTAL]["cost"]

total_processing_time = 0.0
total_cost = 0.0
with open(PERF_FILE, 'w', newline="") as perf_file:
    writer = csv.writer(perf_file)
    writer.writerow(PER_FILE_FIELDS)
    for problem_file in problem_files:
        relative_problem_file = path.relpath(problem_file, getcwd())
        Vm.reset_ids()
        print(f"\n{'-'*50}\nSolving problem {problem_file}\n{'-'*50}")

        # Change to problems directory to load the problem file. The reading of workload files is
        # relative to the problem file location.
        previous_dir = getcwd()
        chdir(PROBLEMS_DIR)
        ascal_config = AscalConfig.get_from_config_yaml(problem_file, aws_eu_west_1_c5m5r5.c5_m5_r5_fm)
        chdir(previous_dir)

        ascal_problem = Ascal(ascal_config)
        timer_start = time()
        ascal_problem.run()
        processing_time = time() - timer_start
        total_processing_time += processing_time
        problem_cost = sum(ascal_problem.get_cluster_cost()) / 3600
        total_cost += problem_cost
        if relative_problem_file in ref_perf_data:
            ref_data = ref_perf_data[relative_problem_file]
            speedup = ref_data["processing_time"] / processing_time if processing_time > 0 else float('inf')
            cost_divider = ref_data["cost"] / problem_cost if problem_cost > 0 else float('inf')
            writer.writerow([relative_problem_file, f" {processing_time:1.2f}", f" {problem_cost:1.2f}",
                             f" {speedup:1.2f}", f" {cost_divider:1.2f}"])
        else:
            writer.writerow([relative_problem_file, f" {processing_time:1.2f}", f" {problem_cost:1.2f}", " -", " -"])
        perf_file.flush()

    # Write the total row
    if path.exists(REF_PERF_FILE):
        total_speedup = total_ref_processing_time / total_processing_time if total_processing_time > 0 else float(' inf')
        total_cost_divider = total_ref_cost / total_cost if total_cost > 0 else float(' inf')
        writer.writerow(["TOTAL", f" {total_processing_time:1.2f}", f" {total_cost:1.2f}", 
                         f" {total_speedup:1.2f}", f" {total_cost_divider:1.2f}"])  
    else:
        writer.writerow(["TOTAL", f" {total_processing_time:1.2f}", f" {total_cost:1.2f}", " -", " -"])  


--------------------------------------------------
Solving problem /home/chechu/ascal-new-features/inplaceverticalpod/perf/../tests/problems/test_020.yaml
--------------------------------------------------
Time: 0 s
You can check the CBC log at /tmp/5b943348b3824321abceb753e4623a33-pulp.lp.log
You can check the CBC log at /tmp/5b8c1b4e95eb4e8abee7d271414711ff-pulp.lp.log
You can check the CBC log at /tmp/4f5001f6ee5947049533529644cf4c2e-pulp.lp.log
You can check the CBC log at /tmp/d30aad599c054602ad548f152953ce89-pulp.lp.log
You can check the CBC log at /tmp/4e4ae470c864481f95b85819025918a0-pulp.lp.log
You can check the CBC log at /tmp/d8a3f9ac9e434eb9bc788b4e45b81705-pulp.lp.log
Time: 100 s
Time: 200 s
Time: 300 s
You can check the CBC log at /tmp/8d44d3cf987e46fab5591ac98d074a11-pulp.lp.log
You can check the CBC log at /tmp/7c5e1e056eaf406da6046a3bfeaccff8-pulp.lp.log
You can check the CBC log at /tmp/adc5399c0ccd4981981bf1c03d92ccb4-pulp.lp.log
You can check the CBC log at /tmp/13